In [1]:
#Ek, erken bitirenler için: modeli trigram'a çevir (iki önceki harfe bak), veriyi train/dev/test olarak %80/%10/%10 böl, 
#smoothing gücünü dev loss'una göre ayarla. Bigram ile trigram loss'unu karşılaştır ve ürettiği isimlerin nasıl değiştiğini göster. 
#Videoda bunun kodu yok, Karpathy videonun sonunda egzersiz olarak veriyor.

In [1]:
import random
import torch

In [2]:
words = open('turkish_names_cleaned.txt', 'r', encoding='utf-8').read().splitlines()
print(len(words))
print(words[:10])

3110
['aba', 'abadan', 'abak', 'abaka', 'abakan', 'abakay', 'abar', 'abasıyanık', 'abay', 'abaza']


In [3]:
def split_dataset(words, train_ratio=0.8, dev_ratio=0.1, seed=42): 
     # a dan z ye sıralandığı için modelin train datalarını %80 ayırdığımız için  z leri görmemesi sorun olabilir bu yüzden shuffle kullanıyoruz.
    
    words_shuffled = words.copy()
    random.seed(seed)
    random.shuffle(words_shuffled)

    n = len(words_shuffled) #3110
    n_train = int(n * train_ratio)  #3110 * 0.8 = 2488
    n_dev = int(n * dev_ratio)   #3110 * 0.1 = 311

    train_words = words_shuffled[:n_train]  #(0 - 2488)
    dev_words = words_shuffled[n_train:n_train + n_dev]  #(2488-2799)
    test_words = words_shuffled[n_train + n_dev:]  #(2799 - 3110)
    return train_words, dev_words, test_words

train_data, dev_data, test_data = split_dataset(words)
print('train:', len(train_data))
print('dev:', len(dev_data))
print('test:', len(test_data))

train: 2488
dev: 311
test: 311


In [4]:
chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)} 
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
print(stoi)

{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'v': 21, 'y': 22, 'z': 23, 'ç': 24, 'ö': 25, 'ü': 26, 'ğ': 27, 'ı': 28, 'ş': 29, '.': 0}


In [5]:
N = torch.zeros((30, 30, 30), dtype=torch.int32)

for w in train_data:
    chs = ['.','.'] + list(w) + ['.']  
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        N[ix1, ix2, ix3] += 1

print(N.shape)
# chs = ['.', 'a', 'h', 'm', 'e', 't', '.'] bigramda bir önceki harfe bakarak bir sonrakini harfi tahmin ediyorduk yani  . ya bakıp a yı a ya bkıp h yi
# trigramda ('.', 'a') İKİSİ BİRDEN, tahmin: 'h'   , ('a', 'h') İKİSİ BİRDEN, tahmin: 'm' yani a hiç tahmin edilmiyor bu yüzden iki tane . . koyuyoruz.


torch.Size([30, 30, 30])


In [6]:
# P = P / P.sum(1, keepdim=True)  bigram da N[i, :] olmak üzere iki eksen vardı 1. sabit 2. si olasılık yani tahmin etmemiz gerekn eksen 2. bu yüzden dim=1 seçiyorduk 
# trigram da ise N[i, j , :]  i ve j sabit ve 3. eksende olasılık yani tahmin etmemiz gerekn eksen 3. bu yüzden dim=2 seçiyoruz indexlerin 0,1,2, gittiğini unutmayalım.

In [7]:
def trigram_loss(words, P, stoi):
    log_likelihood = 0.0
    n = 0.0
    for w in words:
        chs = ['.', '.'] + list(w) + ['.']
        for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
            ix1 = stoi[ch1]
            ix2 = stoi[ch2]
            ix3 = stoi[ch3]
            prob = P[ix1, ix2, ix3]
            logprob = torch.log(prob)
            log_likelihood += logprob
            n += 1
    nll = -log_likelihood
    return (nll / n).item()

In [8]:
smoothing_values = [0.01, 0.1, 0.5, 1, 2, 5, 10, 20, 200, 1000]

best_k = None
best_dev_loss = float('inf')

for k in smoothing_values:
    P = (N + k).float()
    P = P / P.sum(2, keepdim=True)

    dev_loss = trigram_loss(dev_data, P, stoi)
    print(f'k={k}: dev loss = {dev_loss:.4f}')

    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        best_k = k

P = (N + best_k).float()
P = P / P.sum(2, keepdim=True)

print(f'\nen iyi smoothing: k={best_k}, dev loss={best_dev_loss:.4f}')

k=0.01: dev loss = 2.3411
k=0.1: dev loss = 2.2558
k=0.5: dev loss = 2.2943
k=1: dev loss = 2.3591
k=2: dev loss = 2.4583
k=5: dev loss = 2.6348
k=10: dev loss = 2.7862
k=20: dev loss = 2.9346
k=200: dev loss = 3.2749
k=1000: dev loss = 3.3666

en iyi smoothing: k=0.1, dev loss=2.2558


In [9]:
g = torch.Generator().manual_seed(42)

for i in range(20):
    out = []
    ix1, ix2 = 0, 0   # başlangıçta iki nokta . .
    while True:
        p = P[ix1, ix2]
        ix3 = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix3])
        if ix3 == 0:
            break
        ix1, ix2 = ix2, ix3   # bağlam bir ileri gider

    print(''.join(out))

ay.
yiğia.
tayuç.
kökçurulay.
tantayaç.
bağrık.
yunanç.
kan.
adık.
türi.
aliya.
taydiliz.
tungütsagıojığatın.
azıldu.
at.
bınalpez.
han.
satun.
bulak.
nurslakar.


In [10]:
xs1, xs2, ys = [], [], []
for w in train_data:
    chs = ['.', '.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        xs1.append(stoi[ch1])
        xs2.append(stoi[ch2])
        ys.append(stoi[ch3])

xs1 = torch.tensor(xs1)
xs2 = torch.tensor(xs2)
ys = torch.tensor(ys)
num = xs1.nelement()
print('örnek sayısı:', num)

örnek sayısı: 18401


In [11]:
g = torch.Generator().manual_seed(42)
W = torch.randn((2 * 30, 30), generator=g, requires_grad=True)

In [12]:
import torch.nn.functional as F

losses = []

for k in range(2000):
    xenc1 = F.one_hot(xs1, num_classes=30).float()
    xenc2 = F.one_hot(xs2, num_classes=30).float()
    xenc = torch.cat([xenc1, xenc2], dim=1)

    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    loss = -probs[torch.arange(num), ys].log().mean()
    losses.append(loss.item())

    W.grad = None
    loss.backward()
    W.data += -1.0 * W.grad   

print('ilk 5:', losses[:5])
print('son 5:', losses[-5:])

ilk 5: [4.238399982452393, 4.219808101654053, 4.2015838623046875, 4.183718681335449, 4.166202545166016]
son 5: [2.3445656299591064, 2.344510555267334, 2.3444559574127197, 2.3444008827209473, 2.344346046447754]


In [13]:
def nn_trigram_loss(words, W, stoi):
    
    xs1, xs2, ys = [], [], []
    for w in words:
        chs = ['.', '.'] + list(w) + ['.']
        for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
            xs1.append(stoi[ch1]); xs2.append(stoi[ch2]); ys.append(stoi[ch3])
    xs1 = torch.tensor(xs1); xs2 = torch.tensor(xs2); ys = torch.tensor(ys)
    n = xs1.nelement()

    xenc1 = F.one_hot(xs1, num_classes=30).float()
    xenc2 = F.one_hot(xs2, num_classes=30).float()
    xenc = torch.cat([xenc1, xenc2], dim=1)
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    loss = -probs[torch.arange(n), ys].log().mean()
    return loss.item()

print('Sayım modeli (trigram, dev):', best_dev_loss)
print('Sinir ağı (trigram, dev):', nn_trigram_loss(dev_data, W, stoi))

Sayım modeli (trigram, dev): 2.2557647228240967
Sinir ağı (trigram, dev): 2.354698419570923


In [14]:
g2 = torch.Generator().manual_seed(42)

for i in range(20):
    out = []
    ix1, ix2 = 0, 0
    while True:
        xenc1 = F.one_hot(torch.tensor([ix1]), num_classes=30).float()
        xenc2 = F.one_hot(torch.tensor([ix2]), num_classes=30).float()
        xenc = torch.cat([xenc1, xenc2], dim=1)
        logits = xenc @ W
        counts = logits.exp()
        p = counts / counts.sum(1, keepdim=True)

        ix3 = torch.multinomial(p, num_samples=1, replacement=True, generator=g2).item()
        out.append(itos[ix3])
        if ix3 == 0:
            break
        ix1, ix2 = ix2, ix3

    print(''.join(out))

ay.
yiğlatan.
uç.
kögükülekeyileköre.
katerga.
yurağan.
çık.
duulgüne.
ali.
kar.
çe.
emaş.
sın.
tüngrokuçaran.
azık.
saat.
ban.
asız.
hrülügtun.
bölkıkış.


In [15]:
#test datasında sayım modeli için örenkler
g = torch.Generator().manual_seed(22)  

for i in range(20):
    out = []
    ix1, ix2 = 0, 0
    while True:
        p = P[ix1, ix2]
        ix3 = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix3])
        if ix3 == 0:
            break
        ix1, ix2 = ix2, ix3
    print(''.join(out))

bak.
yaıssara.
sarskir.
araaüvdakızda.
cıkan.
kuk.
dan.
keçvkpvvrk.
an.
manyulangüngülü.
yik.
ga.
tumnöüoygandoğlakangingü.
bayarageligeykan.
an.
na.
ertouöıi.
kuşpzdjhfıver.
ayzörük.
töndumtıkaysevgindirkiffvüpa.


In [16]:
# test datası ile gradient descent ile örnekler
g2 = torch.Generator().manual_seed(22)

for i in range(20):
    out = []
    ix1, ix2 = 0, 0
    while True:
        xenc1 = F.one_hot(torch.tensor([ix1]), num_classes=30).float()
        xenc2 = F.one_hot(torch.tensor([ix2]), num_classes=30).float()
        xenc = torch.cat([xenc1, xenc2], dim=1)
        logits = xenc @ W
        counts = logits.exp()
        p = counts / counts.sum(1, keepdim=True)

        ix3 = torch.multinomial(p, num_samples=1, replacement=True, generator=g2).item()
        out.append(itos[ix3])
        if ix3 == 0:
            break
        ix1, ix2 = ix2, ix3
    print(''.join(out))

bak.
yaı.
sara.
sgatkur.
ararakı.
aysabocukandelkudaç.
keçikpz.
ta.
an.
ileyulangü.
bükunyzek.
amagan.
kay.
addoğhayangingü.
bayaraşmek.
kan.
bünç.
na.
ertoungi.
ksık.


In [17]:
test_loss_count = trigram_loss(test_data, P, stoi)
print('Sayım modeli trigram, test datası:', test_loss_count)
test_loss_nn = nn_trigram_loss(test_data, W, stoi)
print('Sinir ağı trigram, test datası:', test_loss_nn)

Sayım modeli trigram, test datası: 2.206176280975342
Sinir ağı trigram, test datası: 2.337057590484619


In [ ]:
print(f'ingilizce veri seti sayım modeli:{2.4588}')
print(f'ingilizce veri seti sinir ağı modeli:{2.4541}')
print(f'türkçe veri seti sayım modeli:{2.4546}')
print(f'türkçe veri seti sinir ağı modeli:{2.4440}')
print(f'türkçe veri seti ile trigram sayım dev:{2.2558}')
print(f'türkçe veri seti ile trigram sinir ağı dev {2.3547}')
print(f'türkçe veri seti ile trigram sayım test:{2.2062}')
print(f'türkçe veri seti ile trigram sinir ağı test:{2.3371}')

ingilizce veri seti sayım modeli:2.4588
ingilizce veri seti sinir ağı modeli:2.4541
türkçe veri seti sayım modeli:2.4546
türkçe veri seti sinir ağı modeli:2.444
türkçe veri seti ile trigram sayım dev:2.2558
türkçe veri seti ile trigram sinir ağı dev 2.3547
türkçe veri seti ile trigram sayım test:2.2062
türkçe veri seti ile trigram sinir ağı test:2.3371
